# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# One row:
#   One row represents a single unique content_id (i.e. page on a website) on a specific date
#   within a month with it's search and engagement metrics calculated for a day.


# Grain:
#   One row per (content_hash_id × report_date)


# Tables Used:
#   Primary table -> fact_content_daily_performance (partitioned by month)
#   Secondary tables (context only) -> dim_content, dim_clients


# Time Window:
#   Development window -> month = '2026-03'
#   Rolling window per row -> Each row shows that specific day's metrics.


# Predict/Rank (label/proxy):
#   This is unsupervised learning. There is no pre-existing label.
#   We have 5 numerical features (impressions, clicks, CTR, position, engagement).
#   K-Means will assign each content_id to one of 4 clusters.
#   Cluster membership is the outcome, created AFTER training.


# Data to deliberately exclude:
#   Query text (anonymized hashes) -> salted pseudonyms that can't be reversed to recover text
#   Rows with zero impressions -> Can't characterize page performance
#   Rows with missing clicks data -> Incomplete activity data
#   Brand and navigational queries (if identifiable) -> Behave differently from informational queries
#   Days with insufficient data -> Very new content may cluster artificially

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Features:
#   1. gsc_impressions -> Daily impressions in Google Search Console
#   2. gsc_clicks	-> Daily clicks from search results
#   3. gsc_avg_position	-> Average daily SERP ranking position
#   4. ga4_pageviews -> Daily pageviews after organic click
#   5. ga4_total_engagement_sec -> Total engagement time per day
#   6. scroll_events -> Daily scroll interactions on page


# Context:
#   1. report_date -> Date of this daily snapshot
#   2. client_hash_id	-> Hashed client identifier
#   3. content_hash_id -> Hashed content identifier
#   4. month -> Calendar month (YYYY-MM)
#   5. gsc_data_available	-> Boolean: GSC data available this day
#   6. ga4_data_available	-> Boolean: GA4 data available this day


# Label:
#   None - unsupervised clustering


# Excluded:
#   1. client_has_gsc
#      Why: Redundant quality flag; use gsc_data_available instead

#   2. client_has_ga4
#      Why: Redundant quality flag; use ga4_data_available instead

#   3. gsc_sum_position
#      Why: Should use avg_position, not sum (sum is meaningless for positions)

#   4. ga4_sessions
#      Why: Overlaps with ga4_engaged_sessions; redundant

#   5. ga4_users
#      Why: Counts unique users; not a direct engagement proxy

#   6. sessions_direct
#      Why: Traffic source detail; adds noise (we care about content quality, not source)

#   7. sessions_referral
#      Why: Traffic source detail; adds noise

#   8. sessions_social
#      Why: Traffic source detail; adds noise

#   9. sessions_paid
#      Why: Traffic source detail; adds noise

#  10. ai_chatgpt
#      Why: AI source breakdowns are too granular; use sessions_ai aggregate instead

#  11. ai_perplexity
#      Why: AI source breakdowns are too granular

#  12. ai_gemini
#      Why: AI source breakdowns are too granular

#  13. ai_copilot
#      Why: AI source breakdowns are too granular

#  14. ai_claude
#      Why: AI source breakdowns are too granular

#  15. ai_meta
#      Why: AI source breakdowns are too granular

#  16. ai_other
#      Why: AI source breakdowns are too granular

#  17. sessions_organic
#      Why: Counts organic sessions; overlaps with impressions/clicks

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Connected to HuggingFace warehouse via DuckDB")

Connected to HuggingFace warehouse via DuckDB


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [14]:
result_schema = con.sql("""
  SELECT *
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
  WHERE month = '2026-03'
  LIMIT 1
""")

df_schema = result_schema.df()
print("Columns in fact_content_daily_performance:")
for col in df_schema.columns:
    print(f"  - {col}")

Columns in fact_content_daily_performance:
  - report_date
  - client_hash_id
  - content_hash_id
  - client_has_gsc
  - client_has_ga4
  - gsc_data_available
  - ga4_data_available
  - gsc_impressions
  - gsc_clicks
  - gsc_sum_position
  - gsc_avg_position
  - ga4_pageviews
  - ga4_sessions
  - ga4_users
  - ga4_engaged_sessions
  - ga4_total_engagement_sec
  - sessions_organic
  - sessions_direct
  - sessions_referral
  - sessions_social
  - sessions_paid
  - sessions_ai
  - ai_chatgpt
  - ai_perplexity
  - ai_gemini
  - ai_copilot
  - ai_claude
  - ai_meta
  - ai_other
  - scroll_events
  - month
